# 05 - Evaluation & Model Comparison

Final comparison of all four trained detectors on the **held-out test set**.

| Model | Architecture type | Backbone |
|---|---|---|
| YOLOv8n | One-stage anchor-free | CSPDarknet (nano) |
| YOLOv8s | One-stage anchor-free | CSPDarknet (small) |
| Faster R-CNN | Two-stage anchor-based | ResNet-50 FPN |
| RetinaNet | One-stage anchor-based | ResNet-50 FPN |

**Steps**
1. Load saved CSVs and build master comparison table  
2. Benchmark YOLO inference speed  
3. Metric bar charts (mAP, Precision, Recall)  
4. Speed comparison (FPS, latency)  
5. Accuracy vs Speed trade-off  
6. Training curve overlay  
7. Radar chart  
8. Architecture analysis & recommendation  

## 1. Setup

In [ ]:
import os, sys, time, glob, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec
from PIL import Image
import torch

IN_COLAB = 'google.colab' in sys.modules

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_DIR  = NOTEBOOK_DIR.parent
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/license-plate-detection')

RESULTS_DIR    = PROJECT_DIR / 'results'
YOLO_RUNS_DIR  = RESULTS_DIR / 'yolo'
MODELS_DIR     = PROJECT_DIR / 'models'
TEST_IMG_DIR   = PROJECT_DIR / 'data' / 'processed' / 'yolo' / 'images' / 'test'
EVAL_DIR       = RESULTS_DIR / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

# Consistent colours and markers for all plots
MODEL_COLORS = {
    'YOLOv8n':     '#2196F3',   # blue
    'YOLOv8s':     '#03A9F4',   # light blue
    'Faster R-CNN':'#FF5722',   # deep orange
    'RetinaNet':   '#4CAF50',   # green
}
MODEL_ORDER = ['YOLOv8n', 'YOLOv8s', 'Faster R-CNN', 'RetinaNet']

print('Setup complete.')
print(f'Project : {PROJECT_DIR}')
print(f'Results : {EVAL_DIR}')

## 2. Load Saved Results

In [ ]:
# ── Load YOLO CSV ─────────────────────────────────────────────────────────────
yolo_csv = YOLO_RUNS_DIR / 'yolov8_comparison.csv'
df_yolo  = pd.read_csv(yolo_csv)
# Normalise column names to a common schema
df_yolo = df_yolo.rename(columns={
    'mAP@50':    'mAP50',
    'mAP@50:95': 'mAP5095',
    'Precision': 'Precision',
    'Recall':    'Recall',
    'Weights (MB)': 'Weights_MB',
})
# Speed will be filled after benchmarking (cell below)
df_yolo['FPS']          = float('nan')
df_yolo['Inference_ms'] = float('nan')
df_yolo['Train_min']    = float('nan')

print('YOLO results:')
print(df_yolo.to_string(index=False))

# ── Load Faster R-CNN / RetinaNet CSV ─────────────────────────────────────────
frcnn_csv = RESULTS_DIR / 'frcnn_retinanet_comparison.csv'
df_frcnn  = pd.read_csv(frcnn_csv)
df_frcnn = df_frcnn.rename(columns={
    'mAP@50 (test)':    'mAP50',
    'mAP@50:95 (test)': 'mAP5095',
    'FPS':              'FPS',
    'Inference (ms)':   'Inference_ms',
    'Training Time (min)': 'Train_min',
})
# Precision/Recall not recorded for these models
df_frcnn['Precision']  = float('nan')
df_frcnn['Recall']     = float('nan')
# Model size from saved .pt files
def model_size_mb(name):
    p = MODELS_DIR / f'{name}_best.pt'
    return round(p.stat().st_size / 1e6, 1) if p.exists() else float('nan')
df_frcnn['Weights_MB'] = df_frcnn['Model'].apply(
    lambda m: model_size_mb('faster_rcnn' if 'Faster' in m else 'retinanet')
)

print('\nFaster R-CNN / RetinaNet results:')
print(df_frcnn.to_string(index=False))

## 3. Benchmark YOLO Inference Speed

We time 50 warm-up + 100 measured runs on the test set to get stable FPS numbers.

In [ ]:
from ultralytics import YOLO

def get_device():
    if torch.backends.mps.is_available(): return 'mps'
    if torch.cuda.is_available():         return '0'
    return 'cpu'

DEVICE = get_device()
print(f'Benchmarking on device: {DEVICE}')

test_images = sorted(glob.glob(str(TEST_IMG_DIR / '*')))
print(f'Test images: {len(test_images)}')


def benchmark_yolo(weight_path, model_label, n_warmup=50, n_measure=100):
    """Return (fps, mean_ms) for one YOLO model."""
    model = YOLO(str(weight_path))
    imgs  = test_images * (n_warmup // len(test_images) + 1)

    # Warm-up
    for p in imgs[:n_warmup]:
        model.predict(p, device=DEVICE, verbose=False)

    # Measure
    imgs_measure = (test_images * (n_measure // len(test_images) + 1))[:n_measure]
    t0 = time.perf_counter()
    for p in imgs_measure:
        model.predict(p, device=DEVICE, verbose=False)
    elapsed = time.perf_counter() - t0

    fps     = n_measure / elapsed
    mean_ms = elapsed / n_measure * 1000
    print(f'  {model_label:12s}  FPS={fps:.1f}   latency={mean_ms:.1f} ms')
    return fps, mean_ms


fps_n, ms_n = benchmark_yolo(MODELS_DIR / 'yolov8n_best.pt', 'YOLOv8n')
fps_s, ms_s = benchmark_yolo(MODELS_DIR / 'yolov8s_best.pt', 'YOLOv8s')

# Write back into df_yolo
df_yolo.loc[df_yolo['Model'] == 'YOLOv8n', ['FPS', 'Inference_ms']] = fps_n, ms_n
df_yolo.loc[df_yolo['Model'] == 'YOLOv8s', ['FPS', 'Inference_ms']] = fps_s, ms_s

## 4. Master Comparison Table

In [ ]:
COLS = ['Model', 'mAP50', 'mAP5095', 'Precision', 'Recall',
        'FPS', 'Inference_ms', 'Train_min', 'Weights_MB']

df_all = pd.concat([df_yolo[COLS], df_frcnn[COLS]], ignore_index=True)

# Fix model order
df_all['Model'] = pd.Categorical(df_all['Model'], categories=MODEL_ORDER, ordered=True)
df_all = df_all.sort_values('Model').reset_index(drop=True)

# Pretty-print
fmt = {
    'mAP50':        '{:.4f}',
    'mAP5095':      '{:.4f}',
    'Precision':    '{:.4f}',
    'Recall':       '{:.4f}',
    'FPS':          '{:.1f}',
    'Inference_ms': '{:.1f}',
    'Train_min':    '{:.1f}',
    'Weights_MB':   '{:.1f}',
}

print('=' * 85)
print('MASTER COMPARISON TABLE  —  Test Set')
print('=' * 85)
display_df = df_all.copy()
for col, f in fmt.items():
    display_df[col] = display_df[col].apply(
        lambda v: f.format(v) if pd.notna(v) else 'N/A'
    )
print(display_df.to_string(index=False))
print('=' * 85)
print('N/A = not recorded for that model')

# Save
df_all.to_csv(EVAL_DIR / 'master_comparison.csv', index=False)
print(f'\nSaved: {EVAL_DIR / "master_comparison.csv"}')

## 5. Metric Bar Charts

In [ ]:
def bar_chart(ax, col, title, df, annotate=True, ylim=(0, 1.15)):
    models = df['Model'].tolist()
    values = df[col].tolist()
    colors = [MODEL_COLORS[m] for m in models]
    bars   = ax.bar(models, values, color=colors, edgecolor='white',
                    linewidth=0.8, alpha=0.9)
    if annotate:
        for bar, v in zip(bars, values):
            if pd.notna(v):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.008,
                        f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(*ylim)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', labelsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    # Highlight best
    best_idx = int(df[col].idxmax()) if df[col].notna().any() else -1
    if best_idx >= 0:
        bars[best_idx].set_edgecolor('gold')
        bars[best_idx].set_linewidth(2.5)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_chart(axes[0], 'mAP50',   'mAP@50  (higher = better)', df_all)
bar_chart(axes[1], 'mAP5095', 'mAP@50:95  (higher = better)', df_all)
fig.suptitle('Detection Accuracy — All Models', fontsize=14)
plt.tight_layout()
p = EVAL_DIR / 'accuracy_map.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_chart(axes[0], 'Precision', 'Precision  (higher = better)', df_all)
bar_chart(axes[1], 'Recall',    'Recall  (higher = better)',    df_all)
fig.suptitle('Precision & Recall — YOLO models only (N/A for others)', fontsize=13)
plt.tight_layout()
p = EVAL_DIR / 'precision_recall.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## 6. Speed Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FPS — higher is better
bar_chart(axes[0], 'FPS', 'Inference Speed  (FPS, higher = better)',
          df_all, ylim=(0, df_all['FPS'].max() * 1.25))
axes[0].set_ylabel('Frames per second')

# Latency — lower is better
models = df_all['Model'].tolist()
values = df_all['Inference_ms'].tolist()
colors = [MODEL_COLORS[m] for m in models]
bars   = axes[1].bar(models, values, color=colors, edgecolor='white',
                     linewidth=0.8, alpha=0.9)
for bar, v in zip(bars, values):
    if pd.notna(v):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 1,
                     f'{v:.1f} ms', ha='center', va='bottom', fontsize=9)
# Highlight fastest (lowest latency)
best_idx = int(df_all['Inference_ms'].idxmin())
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(2.5)
axes[1].set_title('Inference Latency  (ms, lower = better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('ms / image')
axes[1].set_ylim(0, df_all['Inference_ms'].max() * 1.25)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].tick_params(axis='x', labelsize=9)

fig.suptitle('Inference Speed — All Models', fontsize=14)
plt.tight_layout()
p = EVAL_DIR / 'speed_comparison.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## 7. Accuracy vs Speed Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, x_col, x_label in [
    (axes[0], 'FPS',          'Inference Speed (FPS)'),
    (axes[1], 'Inference_ms', 'Inference Latency (ms)'),
]:
    for _, row in df_all.iterrows():
        if pd.isna(row[x_col]) or pd.isna(row['mAP50']):
            continue
        color = MODEL_COLORS[row['Model']]
        # Bubble size proportional to model weights
        sz = row['Weights_MB'] * 12 if pd.notna(row['Weights_MB']) else 200
        ax.scatter(row[x_col], row['mAP50'], s=sz, color=color,
                   alpha=0.85, edgecolors='white', linewidth=1.5, zorder=3)
        offset_x = -0.5 if x_col == 'Inference_ms' else 0.5
        ax.annotate(
            row['Model'],
            (row[x_col], row['mAP50']),
            textcoords='offset points', xytext=(8, 4),
            fontsize=10, fontweight='bold', color=color,
        )

    ax.set_xlabel(x_label, fontsize=11)
    ax.set_ylabel('mAP@50', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

    # Ideal corner annotation
    if x_col == 'FPS':
        ax.set_title('mAP@50 vs Speed (FPS)\n(top-right = ideal)', fontsize=12, fontweight='bold')
    else:
        ax.set_title('mAP@50 vs Latency\n(top-left = ideal)', fontsize=12, fontweight='bold')

fig.suptitle('Accuracy vs Speed Trade-off  (bubble size ∝ model size)', fontsize=13)
plt.tight_layout()
p = EVAL_DIR / 'accuracy_vs_speed.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## 8. Training Curve Overlay (YOLO models)

In [ ]:
def load_yolo_results(run_dir):
    csv = Path(run_dir) / 'results.csv'
    df  = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    return df


df_n = load_yolo_results(YOLO_RUNS_DIR / 'yolov8n')
df_s = load_yolo_results(YOLO_RUNS_DIR / 'yolov8s')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Training Curves — nano vs small', fontsize=14)

pairs = [
    (axes[0, 0], 'train/box_loss',      'Train Box Loss'),
    (axes[0, 1], 'train/cls_loss',      'Train Class Loss'),
    (axes[0, 2], 'train/dfl_loss',      'Train DFL Loss'),
    (axes[1, 0], 'metrics/mAP50(B)',    'Val mAP@50'),
    (axes[1, 1], 'metrics/mAP50-95(B)', 'Val mAP@50:95'),
    (axes[1, 2], 'val/box_loss',        'Val Box Loss'),
]

for ax, col, title in pairs:
    if col in df_n.columns:
        ax.plot(df_n['epoch'], df_n[col],
                color=MODEL_COLORS['YOLOv8n'], linewidth=1.5, label='YOLOv8n')
    if col in df_s.columns:
        ax.plot(df_s['epoch'], df_s[col],
                color=MODEL_COLORS['YOLOv8s'], linewidth=1.5, label='YOLOv8s',
                linestyle='--')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
p = EVAL_DIR / 'yolo_training_curves_overlay.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## 9. Radar Chart — Overall Profile

In [ ]:
# Normalise each dimension to [0, 1] for the radar
# For Inference_ms: invert so that lower latency = higher score

radar_df = df_all[['Model', 'mAP50', 'mAP5095', 'FPS', 'Inference_ms']].copy()

# Fill N/A Precision/Recall with col min so they don't distort the radar
for col in ['mAP50', 'mAP5095', 'FPS', 'Inference_ms']:
    radar_df[col] = pd.to_numeric(radar_df[col], errors='coerce')

def minmax(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn) if mx > mn else series * 0 + 0.5

radar_df['mAP50_n']   = minmax(radar_df['mAP50'])
radar_df['mAP5095_n'] = minmax(radar_df['mAP5095'])
radar_df['Speed_n']   = minmax(radar_df['FPS'])
radar_df['Latency_n'] = 1 - minmax(radar_df['Inference_ms'])  # inverted
# Model size score: inverted (smaller = better)
sizes = df_all['Weights_MB'].fillna(df_all['Weights_MB'].max())
radar_df['Size_n'] = 1 - minmax(sizes)

categories = ['mAP@50', 'mAP@50:95', 'Speed\n(FPS)', 'Low\nLatency', 'Small\nSize']
norm_cols   = ['mAP50_n', 'mAP5095_n', 'Speed_n', 'Latency_n', 'Size_n']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), categories, fontsize=11)

for _, row in radar_df.iterrows():
    values = row[norm_cols].tolist()
    values += values[:1]
    color  = MODEL_COLORS[row['Model']]
    ax.plot(angles, values, color=color, linewidth=2, label=row['Model'])
    ax.fill(angles, values, color=color, alpha=0.08)

ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, color='grey')
ax.grid(color='grey', alpha=0.3)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)
ax.set_title('Model Profile Radar Chart\n(all axes normalised — outer = better)',
             size=13, pad=20, fontweight='bold')

plt.tight_layout()
p = EVAL_DIR / 'radar_chart.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## 10. Architecture Analysis

### 10.1 YOLOv8n — Best Overall

YOLOv8n achieves the **highest mAP@50 (0.9364) and mAP@50:95 (0.5606)** despite being the smallest model (6.3 MB). Several factors explain this:

- **Task simplicity**: License plate detection is a single-class, fixed-aspect-ratio problem. YOLOv8's anchor-free head excels here because it doesn't need complex anchor tuning.
- **Mosaic augmentation**: YOLOv8 trains with mosaic by default, which is particularly effective for small objects in varied backgrounds.
- **Transfer learning fit**: COCO pretrained weights already encode car/vehicle features that transfer directly to plates.

### 10.2 YOLOv8s — Counterintuitively Worse than Nano

YOLOv8s scores **lower** than nano (mAP@50: 0.9011 vs 0.9364). With only 433 images this is expected:

- Larger models have more parameters and **overfit more** on small datasets.
- The capacity gap between nano and small (~3×) is not justified by dataset size.
- Early stopping triggered earlier for small, suggesting the extra capacity was not beneficial.

### 10.3 Faster R-CNN — Strong but Slow

Faster R-CNN (mAP@50: 0.9066) performs comparably to YOLOv8s but at **196 ms latency** (5.1 FPS). The two-stage pipeline (RPN + ROI head) is precise but computationally expensive:

- The Region Proposal Network adds overhead that is unnecessary for a well-defined single-class object.
- Anchor-based matching requires careful tuning for license plate aspect ratios.
- Best suited for applications where accuracy matters more than speed (e.g., batch processing CCTV footage).

### 10.4 RetinaNet — Fastest Two-stage Alternative

RetinaNet (mAP@50: 0.8925) is the lowest performer but also has the **shortest training time** and runs at 7.9 FPS. Focal loss helps with class imbalance but the dataset is already well-balanced (single class, 471 boxes across 433 images), so that advantage disappears.

---

### 10.5 Strengths and Weaknesses Summary

| Model | Strengths | Weaknesses |
|---|---|---|
| **YOLOv8n** | Best accuracy, fastest, smallest weights, easiest deployment | Less interpretable training process |
| **YOLOv8s** | Good ecosystem support | Overfits on small datasets |
| **Faster R-CNN** | Robust two-stage detection, good on dense/small objects | Very slow (196 ms), heavy (167 MB) |
| **RetinaNet** | Shorter training, good for imbalanced datasets | Lowest accuracy here, still slow |

---

### 10.6 Recommendation

**For production deployment → YOLOv8n**  
It is the clear winner: best accuracy, fastest inference, and smallest footprint. The 6.3 MB model can even run on edge devices (Raspberry Pi, phone).

**If dataset grows significantly (>10k images) → revisit YOLOv8s or YOLOv8m**  
The capacity advantage of larger models only materialises with more training data.

**If offline/batch processing with no speed constraint → Faster R-CNN**  
Its precision makes it suitable for forensic or high-confidence scenarios.

## 11. Final Summary Table & Plots

In [ ]:
# One comprehensive 2×3 summary figure
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax_map50  = fig.add_subplot(gs[0, 0])
ax_map95  = fig.add_subplot(gs[0, 1])
ax_fps    = fig.add_subplot(gs[0, 2])
ax_lat    = fig.add_subplot(gs[1, 0])
ax_trtime = fig.add_subplot(gs[1, 1])
ax_size   = fig.add_subplot(gs[1, 2])

bar_chart(ax_map50, 'mAP50',        'mAP@50',                df_all)
bar_chart(ax_map95, 'mAP5095',      'mAP@50:95',             df_all)
bar_chart(ax_fps,   'FPS',          'Speed (FPS ↑)',          df_all,
          ylim=(0, df_all['FPS'].max() * 1.3))
ax_fps.set_ylabel('Frames per second')

# Latency — lower is better, invert highlight logic
models = df_all['Model'].tolist()
for ax, col, title, ylabel, invert in [
    (ax_lat,    'Inference_ms', 'Latency (ms ↓)',         'ms / image',  True),
    (ax_trtime, 'Train_min',    'Training Time (min ↓)',  'minutes',     True),
    (ax_size,   'Weights_MB',   'Model Size (MB ↓)',      'MB',          True),
]:
    values = df_all[col].tolist()
    colors = [MODEL_COLORS[m] for m in models]
    bars   = ax.bar(models, values, color=colors, edgecolor='white',
                    linewidth=0.8, alpha=0.9)
    for bar, v in zip(bars, values):
        if pd.notna(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.02,
                    f'{v:.1f}', ha='center', va='bottom', fontsize=9)
    if invert:
        valid = [v for v in values if pd.notna(v)]
        if valid:
            best_idx = values.index(min(valid))
            bars[best_idx].set_edgecolor('gold')
            bars[best_idx].set_linewidth(2.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max([v for v in values if pd.notna(v)], default=1) * 1.3)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='x', labelsize=9)

fig.suptitle('License Plate Detection — All Models Summary', fontsize=15, fontweight='bold')

# Legend: gold border = best in category
legend_patch = FancyBboxPatch((0, 0), 1, 1, boxstyle='round,pad=0.1',
                               edgecolor='gold', facecolor='none', linewidth=2.5)
fig.legend(handles=[legend_patch], labels=['Best in category'],
           loc='lower center', ncol=1, fontsize=10, framealpha=0.9)

p = EVAL_DIR / 'summary_all_metrics.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

In [ ]:
print('=' * 70)
print('FINAL EVALUATION SUMMARY')
print('=' * 70)
print(display_df.to_string(index=False))
print()

best_map50  = df_all.loc[df_all['mAP50'].idxmax()]
best_map95  = df_all.loc[df_all['mAP5095'].idxmax()]
best_fps    = df_all.loc[df_all['FPS'].idxmax()]
best_lat    = df_all.loc[df_all['Inference_ms'].idxmin()]

print(f'Best mAP@50      → {best_map50["Model"]:15s} ({best_map50["mAP50"]:.4f})')
print(f'Best mAP@50:95   → {best_map95["Model"]:15s} ({best_map95["mAP5095"]:.4f})')
print(f'Fastest (FPS)    → {best_fps["Model"]:15s} ({best_fps["FPS"]:.1f} FPS)')
print(f'Lowest latency   → {best_lat["Model"]:15s} ({best_lat["Inference_ms"]:.1f} ms)')
print()
print('RECOMMENDATION: YOLOv8n — best accuracy AND fastest inference.')
print('=' * 70)
print()
print('Plots saved to:', EVAL_DIR)
for f in sorted(EVAL_DIR.glob('*.png')):
    print(f'  {f.name}')

---
## Project Complete

| Phase | Notebook | Status |
|---|---|---|
| 1 | `01_data_exploration.ipynb` | ✅ Done |
| 2 | `02_data_preprocessing.ipynb` | ✅ Done |
| 3 | `03_train_yolov8.ipynb` | ✅ Done |
| 4 | `04_train_frcnn_retinanet.ipynb` | ✅ Done |
| 5 | `05_evaluation.ipynb` | ✅ Done |